# SarcasTone — Stage 2: Training & 8-Run Modality Ablation Matrix
### Optimized for Google Colab (T4 GPU - 16 GB VRAM)

[![Colab T4](https://img.shields.io/badge/Google_Colab-T4_GPU_Optimized-orange.svg?logo=googlecolab)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.1+-ee4c2c.svg?logo=pytorch)](https://pytorch.org/)

---

### Pipeline Overview
This notebook executes **Stage 2** of SarcasTone:
1. **Mounts Google Drive** and loads pre-extracted feature tensors from `cache_dir`.
2. **Verifies locked deterministic splits** (`splits.json`) ensuring strictly zero leakage.
3. **Runs optional lightweight smoke test** (`scripts/smoke_test.py`) for pipeline health.
4. **Executes the full 8-run modality ablation matrix** on the T4 GPU with mixed precision (`fp16`):
   * `text_only`
   * `speech_only`
   * `video_only`
   * `text_speech_concat` (Baseline naive concatenation)
   * `text_speech_pgin` (Core hypothesis: prosody-gated incongruity)
   * `text_video`
   * `speech_video`
   * `full_multimodal_pgin` (Full Text + Speech Prosody/Acoustics + Video)
5. **Displays side-by-side comparison tables** (Accuracy, Precision, Recall, Macro F1).
6. **Renders comparative visual plots** including multi-metric bar charts and learned gate weight distributions.

## 1. Mount Google Drive
Mount Google Drive to access cached feature tensors and persist training checkpoints and logs.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted successfully at /content/drive")
except ImportError:
    print("ℹ️ Running outside Google Colab environment; skipping Drive mount.")

## 2. Verify Hardware & T4 GPU Environment
Confirm CUDA acceleration and mixed precision (`torch.amp`) support on Colab T4 GPU.

In [ ]:
import sys
import torch

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")

assert torch.cuda.is_available(), "T4 GPU required! Select Runtime > Change runtime type > T4 GPU."

device_name = torch.cuda.get_device_name(0)
vram_bytes = torch.cuda.get_device_properties(0).total_memory
vram_gb = vram_bytes / (1024 ** 3)

print(f"✓ GPU Allocated: {device_name}")
print(f"✓ Total VRAM   : {vram_gb:.2f} GB")
print(f"✓ AMP Support  : fp16 autocast enabled for Colab T4 acceleration.")

## 3. Repository Setup & Dependencies
Set working directory and verify that `src/` modules and `configs/default.yaml` are available.

In [ ]:
import os
import sys
from pathlib import Path

# Candidate repository locations
REPO_DIR = Path("/content/SarcasTone")
DRIVE_REPO_DIR = Path("/content/drive/MyDrive/SarcasTone")

if DRIVE_REPO_DIR.exists() and (DRIVE_REPO_DIR / "configs" / "default.yaml").exists():
    os.chdir(str(DRIVE_REPO_DIR))
    print(f"✓ Working directory: {DRIVE_REPO_DIR}")
elif REPO_DIR.exists() and (REPO_DIR / "configs" / "default.yaml").exists():
    os.chdir(str(REPO_DIR))
    print(f"✓ Working directory: {REPO_DIR}")
else:
    print("Cloning repository into /content/SarcasTone...")
    !git clone https://github.com/your-username/SarcasTone.git /content/SarcasTone
    os.chdir("/content/SarcasTone")
    print("✓ Working directory: /content/SarcasTone")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

!pip install -q -r requirements.txt
print("✓ Dependencies verified.")

## 4. Configuration & Split Verification
Confirm that `splits.json` and cached feature files exist on Drive, and run the verification suite to prove zero leakage.

In [ ]:
import yaml
from pathlib import Path
from scripts.verify_splits import verify_splits, load_config

CONFIG_PATH = Path("configs/default.yaml")
cfg = load_config(CONFIG_PATH)

paths_cfg = cfg.get("paths", {})
cache_dir = Path(paths_cfg.get("cache_dir", "/content/drive/MyDrive/SarcasTone/cache"))
splits_file = Path(paths_cfg.get("splits_file", "/content/drive/MyDrive/SarcasTone/cache/splits.json"))
results_dir = Path(paths_cfg.get("results_dir", "/content/drive/MyDrive/SarcasTone/results"))

results_dir.mkdir(parents=True, exist_ok=True)

print(f"Checking splits file at: {splits_file}")
assert splits_file.is_file(), f"Splits file not found at {splits_file}! Run 01_extract_and_cache_colab.ipynb first."

passed = verify_splits(
    splits_path=splits_file,
    cache_dir=cache_dir,
    check_speaker_disjoint=cfg.get("dataset", {}).get("speaker_disjoint", False),
)
assert passed, "Dataset split verification failed!"
print("✓ Split verification passed with zero leakage across partitions.")

## 5. Lightweight Smoke Test (Sanity Check)
Run `scripts/smoke_test.py` to verify forward passes, incongruity calculation, loss functions, and metrics end-to-end.

In [ ]:
# Run smoke test on GPU to verify models, loss, and metrics
!python scripts/smoke_test.py --device cuda

## 6. Execute the 8-Run Modality Ablation Matrix on T4 GPU
Run `src.experiments.run_ablations` across all 8 configurations on locked splits:
1. `text_only`
2. `speech_only`
3. `video_only`
4. `text_speech_concat` (Baseline naive concatenation)
5. `text_speech_pgin` (Prosody-Gated Incongruity Network)
6. `text_video`
7. `speech_video`
8. `full_multimodal_pgin` (Full Multimodal Network)

Live `tqdm` progress bars display ETA and throughput across the 8 runs, per-run epochs, and training/validation batches.

In [ ]:
import time
from src.experiments.run_ablations import main as ablations_main

print("=" * 75)
print("STARTING 8-RUN MODALITY ABLATION EXPERIMENTS ON T4 GPU")
print("=" * 75)

t0 = time.time()

# Execute all 8 runs sequentially on identical splits with live tqdm progress and ETA
exit_code = ablations_main([
    "--config", str(CONFIG_PATH),
    "--results-dir", str(results_dir),
    "--splits-file", str(splits_file),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
])

elapsed = time.time() - t0
assert exit_code == 0, f"run_ablations failed with exit code: {exit_code}"
print(f"
✓ All 8 ablation runs completed successfully in {elapsed / 60:.2f} minutes!")

## 7. Side-by-Side Comparison Table across All 8 Runs
Load and display the structured results table comparing Accuracy, Precision, Recall, and Macro F1-Score.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

csv_path = results_dir / "ablation_summary.csv"
assert csv_path.is_file(), f"Summary CSV not found at {csv_path}"

df = pd.read_csv(csv_path)

# Selected presentation columns
cols = [
    "run_name", "modalities", "fusion_mode",
    "accuracy", "precision", "recall", "f1",
    "best_val_f1", "best_epoch", "elapsed_s"
]
avail_cols = [c for c in cols if c in df.columns]
display_df = df[avail_cols].copy()

# Rename for clean presentation
display_df.rename(columns={
    "run_name": "Run Name",
    "modalities": "Modalities",
    "fusion_mode": "Fusion Mode",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "Macro F1",
    "best_val_f1": "Best Val F1",
    "best_epoch": "Best Epoch",
    "elapsed_s": "Time (s)",
}, inplace=True)

# Format floats
for col in ["Accuracy", "Precision", "Recall", "Macro F1", "Best Val F1"]:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) and isinstance(x, (int, float)) else str(x))

print("=" * 85)
print("SARCASTONE: 8-RUN MODALITY ABLATION STUDY COMPARISON TABLE")
print("=" * 85)
display(display_df)

# Highlight best run
if "f1" in df.columns:
    best_idx = pd.to_numeric(df["f1"], errors="coerce").idxmax()
    best_row = df.iloc[best_idx]
    print(f"
🏆 Top Performing Model: '{best_row['run_name']}'")
    print(f"   Modalities: {best_row['modalities']}  |  Fusion: {best_row['fusion_mode']}")
    print(f"   Accuracy  : {best_row['accuracy']:.4f}     |  Macro F1: {best_row['f1']:.4f}")

## 8. Comparative Visualizations & Analysis
Generate multi-metric performance bar charts and analyze learned modality gate allocations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set clean styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, ax = plt.subplots(figsize=(15, 6))

plot_df = df.copy()
for col in ["accuracy", "precision", "recall", "f1"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce").fillna(0.0)

x = np.arange(len(plot_df))
width = 0.18

ax.bar(x - 1.5 * width, plot_df["accuracy"],  width, label="Accuracy", color="#3b82f6", edgecolor="black", linewidth=0.5)
ax.bar(x - 0.5 * width, plot_df["precision"], width, label="Precision", color="#10b981", edgecolor="black", linewidth=0.5)
ax.bar(x + 0.5 * width, plot_df["recall"],    width, label="Recall", color="#f59e0b", edgecolor="black", linewidth=0.5)
ax.bar(x + 1.5 * width, plot_df["f1"],        width, label="Macro F1", color="#ef4444", edgecolor="black", linewidth=0.5)

ax.set_ylabel("Score", fontsize=12)
ax.set_title("SarcasTone: 8-Run Modality Ablation Performance on Google Colab T4", fontsize=14, fontweight="bold", pad=15)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["run_name"], rotation=25, ha="right", fontsize=10)
ax.set_ylim(0.0, 1.05)
ax.legend(loc="lower right", frameon=True, fontsize=11)

plt.tight_layout()
chart_path = results_dir / "ablation_comparison_chart.png"
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"✓ Chart saved to: {chart_path}")

### Learned Modality Gating Weights ($g_T, g_A, g_{\text{inc}}, g_V$)
Inspect how PGIN dynamically balances active modalities across the dataset.

In [ ]:
gate_cols = [c for c in ["gate_text", "gate_speech", "gate_inc", "gate_video"] if c in df.columns]
pgin_subset = df[df["fusion_mode"] == "pgin"].dropna(subset=gate_cols, how="all")

if not pgin_subset.empty and gate_cols:
    fig, ax = plt.subplots(figsize=(11, 5))
    gate_data = pgin_subset.set_index("run_name")[gate_cols].fillna(0.0)
    gate_data.columns = ["Text (g_T)", "Speech (g_A)", "Incongruity (g_inc)", "Video (g_V)"]
    gate_data.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        colormap="viridis",
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_ylabel("Mean Gate Weight", fontsize=12)
    ax.set_title("Learned Modality Gate Weights across PGIN Ablation Runs", fontsize=13, fontweight="bold", pad=12)
    ax.set_xticklabels(gate_data.index, rotation=25, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.legend(title="Modality Branch", loc="upper right")
    plt.tight_layout()
    gate_chart_path = results_dir / "gate_weights_chart.png"
    plt.savefig(gate_chart_path, dpi=300)
    plt.show()
    print(f"✓ Gate weight chart saved to: {gate_chart_path}")
else:
    print("ℹ️ Gate weights not available in current runs.")

## 9. Artifact Summary & Saved Checkpoints
Review the artifacts and checkpoints saved in Google Drive.

In [ ]:
print("=" * 65)
print(f"PERSISTED ARTIFACTS ON GOOGLE DRIVE: {results_dir}")
print("=" * 65)

for item in sorted(results_dir.iterdir()):
    if item.is_dir():
        files = [f.name for f in item.iterdir()]
        print(f"📁 {item.name:<26} -> {', '.join(files)}")
    else:
        print(f"📄 {item.name}")

print("=" * 65)
print("✓ All 8 ablation runs complete! Checkpoints and metrics are safely saved to Drive.")
print("=" * 65)